In [39]:
import os
import numpy as np
import pandas as pd
from PIL import Image

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Concatenate, Input
from tensorflow.keras.models import Sequential

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [40]:
# Sample tabular data
df = pd.DataFrame({
    "bedrooms":[2,3],
    "bathrooms":[1,2],
    "price":[200000,350000],
    "image":["dataset/house1.jfif","dataset/house2.jfif"]
})

In [41]:
# CNN model
base_model = MobileNetV2(weights="imagenet", include_top=False, pooling="avg")

/tmp/ipykernel_4011/1477152957.py:2: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(weights="imagenet", include_top=False, pooling="avg")


In [34]:
# Extract image features
image_features = []

for path in df["image"]:
    img = Image.open(path).resize((224,224))
    img = img_to_array(img)
    img = preprocess_input(img)
    img = np.expand_dims(img, axis=0)

    features = base_model.predict(img)
    image_features.append(features.flatten())

image_features = np.array(image_features)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step


In [42]:
# Tabular features
tabular = df[["bedrooms","bathrooms"]].values

In [43]:
# Combine
X = np.concatenate([tabular, image_features], axis=1)
y = df["price"].values

In [44]:
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y)

In [45]:
# Simple regression model
model = Sequential([
    Dense(128, activation="relu", input_shape=(X.shape[1],)),
    Dense(64, activation="relu"),
    Dense(1)
])

model.compile(optimizer="adam", loss="mse")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [46]:
# Train
model.fit(X_train, y_train, epochs=10)

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - loss: 39999619072.0000
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - loss: 39998857216.0000
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 39998144512.0000
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - loss: 39997431808.0000
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss: 39996649472.0000
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - loss: 39995805696.0000
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - loss: 39994925056.0000
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - loss: 39993999360.0000
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss: 39992999936.0000
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - loss: 39991939072.0000


In [47]:
# Predict
preds = model.predict(X_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step


In [48]:
# Metrics
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 349974.6875
RMSE: 349974.68251289264
